# Chapter 12 &mdash; PDA Basics: Finite Control Plus an Unbounded Stack

**Concept 1 of the Chapter 12 decomposition:** *PDA Basics: Finite Control Plus an Unbounded Stack*

An NFA augmented with a stack it fills itself, preloaded with the bottom marker `#`.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-PDA-Basics/Concept-PDA-Basics.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A **pushdown automaton** is an NFA plus a **stack**: unbounded memory with a
last-in-first-out discipline.

Three things to hold onto:

* the stack starts holding **only** the bottom marker $z_0$, written `#` in Jove &mdash;
  nothing is pre-loaded for you;
* the machine **fills the stack itself** as it reads input;
* it is **nondeterministic** by default, and Chapter 11's Concept 17 showed that this
  matters: NPDA are strictly more powerful than DPDA.

The stack is exactly the memory that Chapter 11 showed nesting requires. Everything
else is an NFA.

## 2. Definitions

### A PDA for balanced parentheses

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
Dyck = md2mc('''PDA
!! Push on '(', pop on ')', accept when the input is gone and # is on top.
I : ( , #  ; (#  -> I     !! first '(' -- push it above the bottom marker
I : ( , (  ; ((  -> I     !! another '(' -- push
I : ) , (  ; ''  -> I     !! ')' matches -- pop
I : '' , # ; #   -> F     !! nothing left and stack is just # -- accept
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### What is in the dictionary

In [ ]:
for k in ['Q', 'Sigma', 'Gamma', 'q0', 'z0', 'F']:
    v = Dyck[k]
    print("%-6s : %s" % (k, sorted(v) if isinstance(v, set) else v))

## 3. Tests

Finite control, unbounded stack.

In [ ]:
print("states           :", sorted(Dyck["Q"]), " -- finite")
print("stack alphabet   :", sorted(Dyck["Gamma"]))
print("bottom marker    :", Dyck["z0"])
assert len(Dyck["Q"]) == 2

It recognises balanced parentheses, to any depth the stack limit allows.

In [ ]:
def balanced(s):
    d = 0
    for ch in s:
        d += 1 if ch == '(' else -1
        if d < 0: return False
    return d == 0

for s in ['', '()', '(())', '()()', '((()))', '(', ')(', '())']:
    got = pda_accepts(Dyck, s, STKMAX=8)
    print("  %-9r balanced %-6s PDA %s" % (s, balanced(s), got))
    assert got == balanced(s)

The stack grows with the nesting depth &mdash; that is the unbounded memory.

In [ ]:
for n in range(1, 5):
    s = '(' * n + ')' * n
    surv, paths, visited = run_pda(s, Dyck, STKMAX=10)
    deepest = max(len(st) for (_, _, st) in visited)
    print("  depth %d : deepest stack seen %d" % (n, deepest))
assert pda_accepts(Dyck, '((((()))))', STKMAX=12)

Contrast with a DFA, whose memory is fixed once and for all.

In [ ]:
print("DFA : |Q| states, chosen at design time -- bounded")
print("PDA : |Q| states PLUS a stack of any height -- unbounded")
print()
print("That single addition takes you from regular to context-free.")

## 4. Animation

Watch the stack rise and fall as the brackets nest.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(Dyck, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. What happens if you forget the `#` bottom marker?
2. How would you make the machine accept by **empty stack** instead?
3. Is this PDA deterministic? Check every (state, input, stack-top) triple.

In [ ]:
# Your work for the exercises above.